# Project 14 — Local Financial Report Analyst

## QA Over Annual Reports with Table + Text Parsing

**Goal:** Build a local RAG system that combines **tabular financial data** (income
statements, balance sheets) with **narrative text** (management discussion, risk
factors) to answer financial analysis questions — all running locally.

**Stack:** Ollama · LangChain · ChromaDB · pandas · Jupyter

```
CSV tables ──► pandas ──► table-to-text ─┐
                                         ├──► chunks ──► Ollama Embeddings ──► ChromaDB
Narrative text ──► Document ─────────────┘                                       │
                                                                                 │
Financial question ──► embed ──► retrieval ──► top chunks ──► LLM ──► Analysis
```

### What You'll Learn

1. Convert structured financial tables to natural language for indexing
2. Combine tabular and narrative data in one vector store
3. Build a financial analyst prompt with calculation support
4. Compute key financial metrics with pandas
5. Generate LLM-powered trend analysis

### Prerequisites

- **Ollama** running with `nomic-embed-text` and `qwen3:8b`
- Python 3.9+

In [ ]:
# Install dependencies (uncomment and run once)
!pip install -q langchain langchain-ollama langchain-community langchain-text-splitters chromadb pandas

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests

OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama is running — {len(models)} model(s) available")
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE}: {e}")

## Step 2 — Configure LLM and Embeddings

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

LLM_MODEL = "qwen3:8b"
EMBED_MODEL = "nomic-embed-text"

llm = ChatOllama(model=LLM_MODEL, temperature=0)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

vec = embeddings.embed_query("test")
print(f"✅ Embedding model ready — dimension: {len(vec)}")
resp = llm.invoke("Say hello.")
print(f"✅ LLM ready — response: {resp.content[:80]}")

## Step 3 — Create Sample Financial Data

We create two types of financial content:
1. **Structured data** — an income statement as a pandas DataFrame (CSV)
2. **Narrative text** — the management discussion from an annual report

This dual-source approach is realistic: real 10-K filings contain both.

In [ ]:
import pandas as pd
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

# Income statement (structured)
income_data = {
    "Year": [2022, 2023, 2024],
    "Revenue_M": [450, 580, 720],
    "COGS_M": [180, 220, 270],
    "Gross_Profit_M": [270, 360, 450],
    "Operating_Expenses_M": [150, 180, 210],
    "Net_Income_M": [85, 128, 175],
}
df = pd.DataFrame(income_data)
df.to_csv("sample_data/income_statement.csv", index=False)
print("Income Statement:")
print(df.to_string(index=False))

# Narrative report (unstructured)
narrative = """Annual Report 2024 — TechCorp Inc.

Financial Highlights:
Revenue grew 24% year-over-year to $720M, driven by enterprise SaaS expansion
and the successful launch of our AI analytics platform. Gross margins improved
to 62.5% from 62.1% as we optimized cloud infrastructure costs. Net income
reached $175M, a 37% increase, reflecting strong operating leverage.

Strategic Initiatives:
- Launched AI-powered analytics platform (Q2 2024), contributing $45M in new ARR
- Expanded to 3 new international markets (Germany, Japan, Brazil)
- Acquired DataViz Corp for $50M to enhance visualization capabilities
- Hired 200 engineers, growing R&D team to 800 people

Risk Factors:
- Enterprise sales cycles lengthening from 45 to 60 days on average
- Currency headwinds expected to impact 2025 revenue by 2-3%
- Key competitor launched similar AI features at a lower price point
- Increasing regulatory scrutiny around AI data processing in EU

Outlook for 2025:
We expect revenue of $870-$910M (21-26% growth). Operating margins should
expand to 30-32% as we achieve scale. We plan to invest $120M in R&D,
focusing on AI capabilities and international expansion.
"""

Path("sample_data/annual_report.txt").write_text(narrative, encoding="utf-8")
print("\n✅ Financial data and narrative report created")

## Step 4 — Convert Table to Natural Language

**Why?** Embedding models understand natural language better than raw CSV rows.
We convert each row of the income statement into a readable sentence. This
"table-to-text" step is critical for mixed tabular+text RAG systems.

In [ ]:
from langchain_core.documents import Document

# Convert table rows to natural language
table_lines = ["Income Statement Summary for TechCorp Inc.:\n"]
for _, row in df.iterrows():
    year = int(row["Year"])
    rev = row["Revenue_M"]
    gp = row["Gross_Profit_M"]
    ni = row["Net_Income_M"]
    gm = gp / rev * 100
    nm = ni / rev * 100
    table_lines.append(
        f"Year {year}: Revenue ${rev}M, Gross Profit ${gp}M "
        f"({gm:.1f}% margin), Net Income ${ni}M ({nm:.1f}% margin)"
    )
table_text = "\n".join(table_lines)

print("Table as natural language:")
print(table_text)

## Step 5 — Build Combined Text + Table Index

We create LangChain `Document` objects for both the narrative and the
table-as-text, then chunk and index them together in ChromaDB.

Each document has `type` metadata to distinguish sources.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
import shutil

narrative_doc = Document(
    page_content=Path("sample_data/annual_report.txt").read_text(encoding="utf-8"),
    metadata={"type": "narrative", "source": "annual_report_2024"}
)

table_doc = Document(
    page_content=table_text,
    metadata={"type": "financial_table", "source": "income_statement"}
)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
all_chunks = splitter.split_documents([narrative_doc, table_doc])

CHROMA_DIR = "sample_data/financial_chroma"
if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)

vectorstore = Chroma.from_documents(
    all_chunks, embeddings,
    persist_directory=CHROMA_DIR, collection_name="financial"
)
print(f"✅ Indexed {len(all_chunks)} chunks (narrative + tables)")
for c in all_chunks:
    print(f"   [{c.metadata['type']:16s}] {len(c.page_content):3d} chars | {c.page_content[:60]}...")

## Step 6 — Test Retrieval

Verify the retriever fetches the right source type for different questions.

In [ ]:
test_qs = [
    "What was the revenue in 2024?",
    "What are the risk factors?",
    "What is the gross margin trend?",
]

for q in test_qs:
    print(f"\nQuery: '{q}'")
    results = vectorstore.similarity_search_with_score(q, k=2)
    for i, (doc, score) in enumerate(results):
        print(f"   [{i+1}] score={score:.4f} | {doc.metadata['type']:16s} | {doc.page_content[:70]}...")

## Step 7 — Build the Financial Analyst Chain

The prompt instructs the LLM to:
- Use ONLY the provided financial data
- Show calculations when citing numbers
- Distinguish between table data and narrative discussion

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

system_prompt = (
    "You are a financial analyst assistant. Answer questions about TechCorp Inc. "
    "using the provided financial data and reports.\n\n"
    "Rules:\n"
    "1. When citing numbers, be precise and show calculations where applicable.\n"
    "2. Distinguish between data from financial tables and narrative discussion.\n"
    "3. If the data doesn't support an answer, say so explicitly.\n"
    "4. Provide year-over-year comparisons when relevant.\n\n"
    "Financial Data:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def financial_query(question: str) -> dict:
    """Query the financial analyst."""
    docs = retriever.invoke(question)
    context_text = "\n\n---\n\n".join(
        f"[Source: {d.metadata['type']}]\n{d.page_content}" for d in docs
    )
    response = (prompt | llm | StrOutputParser()).invoke(
        {"context": context_text, "input": question}
    )
    return {"answer": response, "sources": docs}


print("✅ Financial analyst chain ready!")

## Step 8 — Test Financial Q&A

Let's ask a mix of quantitative and qualitative questions.

In [ ]:
questions = [
    "What was the revenue growth rate from 2023 to 2024?",
    "What are the main risk factors for 2025?",
    "How much did the AI analytics platform contribute in ARR?",
    "Calculate the net income margin trend over 3 years.",
    "What is the revenue guidance for 2025?",
    "How many engineers does TechCorp employ?",
]

for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    result = financial_query(q)
    print(f"\nA: {result['answer']}")
    source_types = set(s.metadata['type'] for s in result['sources'])
    print(f"\n📊 Sources used: {source_types}")

## Step 9 — Compute Key Financial Metrics with pandas

Beyond RAG, we use pandas to compute derived metrics and ask the LLM
to interpret the trends. This combines quantitative analysis with
qualitative reasoning.

In [ ]:
# Compute key metrics
df["Gross_Margin_Pct"] = (df["Gross_Profit_M"] / df["Revenue_M"] * 100).round(1)
df["Net_Margin_Pct"] = (df["Net_Income_M"] / df["Revenue_M"] * 100).round(1)
df["Revenue_Growth_Pct"] = df["Revenue_M"].pct_change().mul(100).round(1)
df["OpEx_Ratio_Pct"] = (df["Operating_Expenses_M"] / df["Revenue_M"] * 100).round(1)

print("Key Financial Metrics:")
print(df[["Year", "Revenue_M", "Revenue_Growth_Pct", "Gross_Margin_Pct",
          "Net_Margin_Pct", "OpEx_Ratio_Pct"]].to_string(index=False))

## Step 10 — LLM-Powered Trend Analysis

We feed the computed metrics to the LLM and ask for a trend interpretation.
This demonstrates combining programmatic analysis with LLM reasoning.

In [ ]:
trend_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a financial analyst. Analyze the key financial trends "
               "from the data provided. Focus on: revenue growth trajectory, "
               "margin expansion/compression, and operating efficiency."),
    ("human", "Analyze these financial metrics for TechCorp Inc.:\n\n{data}"),
])

trend_chain = trend_prompt | llm | StrOutputParser()
analysis = trend_chain.invoke({"data": df.to_string()})
print("📈 Trend Analysis:")
print(analysis)

## Step 11 — Interactive Financial Helper

In [ ]:
def ask(question: str) -> str:
    """Ask the financial analyst."""
    result = financial_query(question)
    print(f"Answer:\n{result['answer']}\n")
    print(f"Sources ({len(result['sources'])} chunks):")
    for i, doc in enumerate(result['sources']):
        print(f"   [{i+1}] {doc.metadata['type']}: {doc.page_content[:80]}...")
    return result['answer']

_ = ask("What acquisition did TechCorp make and how much did it cost?")

## Limitations & Tradeoffs

| Aspect | What happens | How to improve |
|--------|-------------|----------------|
| **Table parsing** | We manually converted CSV to text | Use specialized table extractors for real PDFs |
| **Calculation accuracy** | LLM may make arithmetic errors | Offload calculations to pandas, use LLM for interpretation |
| **Multi-year analysis** | Only 3 years of data | Add more historical data |
| **Complex formulas** | LLM struggles with multi-step calculations | Use code generation for complex finance math |
| **Real filings** | Simulated data | Parse real SEC EDGAR filings |

## What You Learned

1. **Table-to-text conversion** — making structured data searchable via embeddings
2. **Mixed-source RAG** — combining tables and narratives in one vector store
3. **Financial prompting** — instructing the LLM to show calculations
4. **pandas metrics** — computing derived financial ratios programmatically
5. **LLM trend analysis** — combining quantitative data with qualitative reasoning

## Exercises

1. **Add a balance sheet** — create balance sheet data and index it alongside
2. **Multi-company comparison** — add a second company and compare metrics
3. **Chart generation** — plot revenue and margin trends with matplotlib
4. **Real 10-K filing** — download a real SEC filing and parse it
5. **Ratio calculations** — add ROE, ROA, debt-to-equity and ask the LLM to interpret

---

*Next project: **15 — Local Contract Clause Finder***